# Evaluation of Data Scientist Agents for Finance

## Load results

In [1]:
import pandas as pd
import json
import numpy as np

gold_standard = pd.read_csv(
    "/Users/thiagocastroferreira/Desktop/kubernetes/mcp-tutorial/fundamental_analysis/2025-04-17/fundamental_analysis.csv"
)

terms = pd.read_csv("fundamental_analysis/terms.csv")["Termo"].tolist() + [
    "Receita Líquida (3 meses)",
    "EBIT (3 meses)",
    "Lucro Líquido (3 meses)",
]
for i, term in enumerate(terms):
    if term in ["Receita Líquida", "EBIT", "Lucro Líquido"]:
        terms[i] = f"{term} (12 meses)"

ENERGY_COMPANIES = [
    {"id": "ALUP11", "cnpj": "08.364.948/0001-38", "name": "Alupar Investimento"},
    {"id": "AURE3", "cnpj": "28.594.234/0001-23", "name": "Auren Energia"},
    {"id": "CPLE3", "cnpj": "76.483.817/0001-20", "name": "Companhia Paranaense de Energia"},
    {"id": "EGIE3", "cnpj": "02.474.103/0001-19", "name": "Engie Brasil Energia"},
    {"id": "ELET3", "cnpj": "00.001.180/0001-26", "name": "Eletrobrás"},
    {"id": "ENEV3", "cnpj": "04.423.567/0001-21", "name": "Eneva"},
    {"id": "ENGI3", "cnpj": "00.864.214/0001-06", "name": "Energisa"},
    {"id": "EQTL3", "cnpj": "03.220.438/0001-73", "name": "Equatorial"},
    {"id": "ISAE3", "cnpj": "02.998.611/0001-04", "name": "ISA Energia Brasil"},
    {"id": "LIGT3", "cnpj": "03.378.521/0001-75", "name": "Light"},
    {"id": "NEOE3", "cnpj": "01.083.200/0001-18", "name": "Neoenergia"},
    {"id": "RNEW11", "cnpj": "08.534.605/0001-74", "name": "Renova Energia"},
    {"id": "SRNA3", "cnpj": "42.500.384/0001-51", "name": "Serena Energia"},
]
stocks = [s["id"] for s in ENERGY_COMPANIES]

gold_standard.rename(
    columns={
        "Receita Líquida": "Receita Líquida (12 meses)",
        "EBIT": "EBIT (12 meses)",
        "Lucro Líquido": "Lucro Líquido (12 meses)",
    },
    inplace=True,
)
gold_standard = gold_standard[gold_standard["Papel"].isin(stocks)].sort_values(by="Papel")

In [2]:
gold_standard[["Papel", "P/L", "P/VP", "LPA", "ROIC", "Cotação", "Patrim. Líq", "Nro. Ações"]]

,Papel,P/L,P/VP,LPA,ROIC,Cotação,Patrim. Líq,Nro. Ações
79,ALUP11,8.69,1.15,3.29,10.1,28.63,8.240130e+09,9.888810e+08
87,AURE3,33.07,0.63,0.24,2.2,7.90,1.320170e+10,1.050380e+09
81,CPLE3,10.57,1.16,0.94,7.7,9.96,2.567470e+10,2.982810e+09
78,EGIE3,7.49,2.85,5.24,12.4,39.29,1.126670e+10,8.159280e+08
80,ELET3,9.37,0.80,4.50,5.4,42.13,1.218630e+11,2.307100e+09
88,ENEV3,573.62,1.27,0.02,6.1,12.47,1.890950e+10,1.932590e+09
77,ENGI3,7.25,1.59,1.66,9.9,12.00,1.727950e+10,2.289420e+09
84,EQTL3,15.23,1.64,2.24,9.7,34.15,2.611370e+10,1.253850e+09
75,ISAE3,6.01,1.07,5.31,11.4,31.91,1.971460e+10,6.588830e+08
71,LIGT3,1.06,0.33,4.41,7.0,4.67,5.218460e+09,3.725550e+08


## Mean Absolute Error Formula

Clipping error to 100

In [3]:
def calculate_mae(y_true: list, y_pred: list, alpha: float = 10, zero_tol: float = 0.0):
    """
    Calcula um score composto:
      score = NMAE + alpha * penalty_rate

    penalty_rate: fração de previsões zero indevidas
    """

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Mantém apenas valores válidos
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan

    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask]

    # --- NMAE ---
    range_y = y_true_f.max() - y_true_f.min()
    if range_y == 0:
        return np.nan

    abs_err = np.abs(y_true_f - y_pred_f)
    nmae = abs_err.mean() / range_y

    # --- Penalidade por colapso em zero ---
    # zero_tol permite tratar valores "quase zero" como zero
    zero_pred = np.abs(y_pred_f) <= zero_tol
    nonzero_true = np.abs(y_true_f) > zero_tol

    penalty_rate = np.mean(zero_pred & nonzero_true)

    # --- Score final ---
    score = nmae + alpha * penalty_rate

    return score

## Calculating Error

Calculating error for framework, task and server

In [4]:
PATH = "/Users/thiagocastroferreira/Desktop/kubernetes/results/final_report2025_2"

settings = {
    "gpt-4.1-nano": {
        "agent": [True, False],
        "workflow": [True, False],
    },
    "gpt-4.1-mini": {
        "agent": [True, False],
        "workflow": [True, False],
    },
    "gpt-5-nano": {
        "agent": [True, False],
        "workflow": [True, False],
    },
    "gpt-5-mini": {
        "agent": [True, False],
        "workflow": [True, False],
    },
}

models = []
for model in settings:
    for architecture in settings[model]:
        for reflection in settings[model][architecture]:
            path = f"{PATH}/{model}/{architecture}_{reflection}"

            df = []

            for stock in stocks:
                indicators = {"total_tokens": [], "time_seconds": []}
                for run_idx in range(3):
                    with open(f"{path}/{stock}_output_{run_idx}.json", "r") as f:
                        output = json.load(f)

                    with open(f"{path}/{stock}_{run_idx}.json", "r") as f:
                        metadata = json.load(f)

                    for i in output["indicators"]:
                        indicator, indicator_value = i["indicator"], i["value"]
                        if indicator not in indicators:
                            indicators[indicator] = []
                        indicators[indicator].append(indicator_value)

                    indicators["total_tokens"].append(metadata["usage"]["total_tokens"])
                    indicators["time_seconds"].append(metadata["time"])

                for key in indicators:
                    indicators[key] = float(np.mean(indicators[key]))
                indicators["stock"] = stock
                indicators["model"] = model
                indicators["architecture"] = architecture
                indicators["reflection"] = reflection

                for term in terms:
                    if term not in indicators:
                        indicators[term] = 0
                df.append(indicators)

            df = pd.DataFrame(df)
            models.append((f"{model}/{architecture}/{reflection}", df))

In [5]:
from tabulate import tabulate

headers = ["Termo", "Model", "MAE"]
results = []
results_df = {h: [] for h in headers}
for term in terms:
    for model_name, df in models:
        y_pred = df[term].to_list()
        y_gold = gold_standard[term].to_list()

        if len(y_gold) == len(y_pred):
            mae = calculate_mae(y_pred=y_pred, y_true=y_gold)

            results_df["Termo"].append(term)
            results_df["Model"].append(model_name)
            results_df["MAE"].append(mae)

# print(tabulate(results, headers=headers, tablefmt="grid"))
results_df = pd.DataFrame(results_df)
results_df

,Termo,Model,MAE
0,Ativo,gpt-4.1-nano/agent/True,8.607666e-01
1,Ativo,gpt-4.1-nano/agent/False,8.696968e-01
2,Ativo,gpt-4.1-nano/workflow/True,4.287402e-02
3,Ativo,gpt-4.1-nano/workflow/False,7.291805e-03
4,Ativo,gpt-4.1-mini/agent/True,6.072026e-03
...,...,...,...
507,Lucro Líquido (3 meses),gpt-5-nano/workflow/False,1.298626e+297
508,Lucro Líquido (3 meses),gpt-5-mini/agent/True,2.509127e-02
509,Lucro Líquido (3 meses),gpt-5-mini/agent/False,3.633349e-02
510,Lucro Líquido (3 meses),gpt-5-mini/workflow/True,2.369432e-02


In [6]:
headers = ["Model", "MAE", "token/seconds", "Time (seconds)", "Tokens"]
results = []
metadata_df = {h: [] for h in headers}
for model_name, df in models:
    metadata_df["Model"].append(model_name)
    time_seconds = df["time_seconds"].mean()
    ntokens = df["total_tokens"].mean()
    token_seconds = ntokens / time_seconds
    metadata_df["Time (seconds)"].append(round(time_seconds, 2))
    metadata_df["Tokens"].append(round(ntokens, 2))
    metadata_df["token/seconds"].append(round(token_seconds, 2))
    metadata_df["MAE"].append(0)

# print(tabulate(results, headers=headers, tablefmt="grid"))
metadata_df = pd.DataFrame(metadata_df)

Evaluation of Models

In [7]:
grouped = results_df[["Model", "MAE"]].groupby(by=["Model"])

for group_keys, group_df in grouped:
    r = []
    for mae_list in group_df["MAE"]:
        r += [mae_list]
    mae = round(sum(r) / len(r), 2)
    print(group_keys, mae)
    metadata_df.loc[metadata_df["Model"] == group_keys[0], "MAE"] = float(mae)

('gpt-4.1-mini/agent/False',) 2.16
('gpt-4.1-mini/agent/True',) 1.57
('gpt-4.1-mini/workflow/False',) 159406.23
('gpt-4.1-mini/workflow/True',) 1.27
('gpt-4.1-nano/agent/False',) 70.2
('gpt-4.1-nano/agent/True',) 3.6
('gpt-4.1-nano/workflow/False',) 234389.25
('gpt-4.1-nano/workflow/True',) 1.19
('gpt-5-mini/agent/False',) 1.0
('gpt-5-mini/agent/True',) 1.01
('gpt-5-mini/workflow/False',) 0.43
('gpt-5-mini/workflow/True',) 0.63
('gpt-5-nano/agent/False',) 1.72
('gpt-5-nano/agent/True',) 0.66
('gpt-5-nano/workflow/False',) 8.087971310863223e+295
('gpt-5-nano/workflow/True',) 9.800310224842642e+17


/var/folders/1p/jbswfpbs73q5qbbh78dzj5xm0000gn/T/ipykernel_93769/3698071695.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2.16' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  metadata_df.loc[metadata_df["Model"] == group_keys[0], "MAE"] = float(mae)


In [8]:
from IPython.display import display, Markdown

idf = metadata_df.sort_values(by=["Model"])
headers = ["Model", "Architecture", "Reflect", "MAE", "Tokens"]
tabresults = []
for idx, row in idf.iterrows():
    model, architecture, reflection = row["Model"].split("/")
    tabresults.append([model, architecture, reflection, row["MAE"], round(row["Tokens"])])

print(tabulate(tabresults, headers=headers))
latex_table = tabulate(tabresults, headers=headers, tablefmt="latex")
display(Markdown(f"```latex\n{latex_table}\n```"))

Model         Architecture    Reflect                  MAE    Tokens
------------  --------------  ---------  -----------------  --------
gpt-4.1-mini  agent           False           2.16            172502
gpt-4.1-mini  agent           True            1.57            195826
gpt-4.1-mini  workflow        False      159406               193569
gpt-4.1-mini  workflow        True            1.27            215343
gpt-4.1-nano  agent           False          70.2              85016
gpt-4.1-nano  agent           True            3.6             135225
gpt-4.1-nano  workflow        False      234389               159497
gpt-4.1-nano  workflow        True            1.19            224625
gpt-5-mini    agent           False           1               198611
gpt-5-mini    agent           True            1.01            218438
gpt-5-mini    workflow        False           0.43            204769
gpt-5-mini    workflow        True            0.63            246836
gpt-5-nano    agent           Fals

```latex
\begin{tabular}{lllrr}
\hline
 Model        & Architecture   & Reflect   &               MAE &   Tokens \\
\hline
 gpt-4.1-mini & agent          & False     &      2.16         &   172502 \\
 gpt-4.1-mini & agent          & True      &      1.57         &   195826 \\
 gpt-4.1-mini & workflow       & False     & 159406            &   193569 \\
 gpt-4.1-mini & workflow       & True      &      1.27         &   215343 \\
 gpt-4.1-nano & agent          & False     &     70.2          &    85016 \\
 gpt-4.1-nano & agent          & True      &      3.6          &   135225 \\
 gpt-4.1-nano & workflow       & False     & 234389            &   159497 \\
 gpt-4.1-nano & workflow       & True      &      1.19         &   224625 \\
 gpt-5-mini   & agent          & False     &      1            &   198611 \\
 gpt-5-mini   & agent          & True      &      1.01         &   218438 \\
 gpt-5-mini   & workflow       & False     &      0.43         &   204769 \\
 gpt-5-mini   & workflow       & True      &      0.63         &   246836 \\
 gpt-5-nano   & agent          & False     &      1.72         &    82059 \\
 gpt-5-nano   & agent          & True      &      0.66         &    65242 \\
 gpt-5-nano   & workflow       & False     &      8.08797e+295 &   223107 \\
 gpt-5-nano   & workflow       & True      &      9.80031e+17  &   260114 \\
\hline
\end{tabular}
```

In [9]:
metadata_df.sort_values(by=["MAE"])

,Model,MAE,token/seconds,Time (seconds),Tokens
15,gpt-5-mini/workflow/False,4.300000e-01,1404.78,145.77,204769.46
14,gpt-5-mini/workflow/True,6.300000e-01,1441.16,171.28,246835.79
8,gpt-5-nano/agent/True,6.600000e-01,504.14,129.41,65241.92
13,gpt-5-mini/agent/False,1.000000e+00,1504.66,132.00,198611.03
12,gpt-5-mini/agent/True,1.010000e+00,1443.41,151.33,218437.72
2,gpt-4.1-nano/workflow/True,1.190000e+00,10334.76,21.73,224625.41
6,gpt-4.1-mini/workflow/True,1.270000e+00,2233.75,96.40,215342.95
4,gpt-4.1-mini/agent/True,1.570000e+00,3236.60,60.50,195825.79
9,gpt-5-nano/agent/False,1.720000e+00,701.64,116.95,82058.85
5,gpt-4.1-mini/agent/False,2.160000e+00,3196.74,53.96,172501.87


Total tokens

In [71]:
grouped = results_df[["Model", "MAE"]].groupby(by=["Model"])

for group_keys, group_df in grouped:
    r = []
    for mae_list in group_df["MAE"]:
        r.extend(mae_list)
    print(group_keys, round(sum(r) / len(r), 2))

TypeError: 'float' object is not iterable

Evaluation of Models per Term

In [72]:
import numpy as np
from tabulate import tabulate

headers = ["Termo", "Workflow 5-mini", "Agent 5-nano Reflect"]
grouped = results_df[["Model", "Termo", "MAE"]].groupby(by=["Model", "Termo"])

terms_mae = {}
map2model = {
    "gpt-5-mini/workflow/False": "Workflow 5-mini",
    "gpt-5-nano/agent/True": "Agent 5-nano Reflect",
}
for group_keys, group_df in grouped:
    r = []
    for mae_list in group_df["MAE"]:
        r += [mae_list]
    model, term = group_keys
    mae = round(sum(r) / len(r), 2)
    if term not in terms_mae:
        terms_mae[term] = {}

    try:
        terms_mae[term][map2model[model]] = mae
    except Exception:
        pass

results = []
for term in terms_mae:
    results.append(
        [
            term,
            terms_mae[term]["Workflow 5-mini"],
            terms_mae[term]["Agent 5-nano Reflect"],
        ]
    )

latex_table = tabulate(results, headers=headers, tablefmt="latex")
print(latex_table)

\begin{tabular}{lrr}
\hline
 Termo                      &   Workflow 5-mini &   Agent 5-nano Reflect \\
\hline
 Ativo                      &              0    &                   0.21 \\
 Ativo Circulante           &              0    &                   0.19 \\
 Disponibilidades           &              0.01 &                   0.17 \\
 Div Br/ Patrim             &              1.15 &                   5.49 \\
 Dív. Bruta                 &              0    &                   0.38 \\
 Dív. Líquida               &              0.01 &                   0.52 \\
 EBIT (12 meses)            &              0.09 &                   0.25 \\
 EBIT (3 meses)             &              0.07 &                   0.38 \\
 EBIT / Ativo               &              0.11 &                   0.37 \\
 EV / EBIT                  &              0.26 &                   0.34 \\
 EV / EBITDA                &              0.08 &                   0.11 \\
 Giro Ativos                &              0.85 &    

Checking examples

# Manager

### Proposed Approach

Cumulative Return

In [78]:
import json
import numpy as np

FOLDER = "/Users/thiagocastroferreira/Desktop/kubernetes/results/manager"

with open(f"{FOLDER}/decisions_sample.json") as f:
    decisions = json.load(f)

with open(f"{FOLDER}/results_sample.json") as f:
    results = json.load(f)

# Store open positions and realized trade returns
open_positions = {}
trade_returns_per_stock = {}
last_cotation_per_stock = {}

for i, decision in enumerate(decisions):
    stock_id = decision["stock_id"]
    analysis_date = decision["analysis_date"]
    recommendation = decision["recommendation"]
    cotation = [
        r for r in results if r["ACAO"] == stock_id and r["DATA_DO_PREGAO"] == analysis_date
    ][0]["PRECO_ULTIMO_NEGOCIO"]
    last_cotation_per_stock[stock_id] = cotation

    if stock_id not in open_positions:
        open_positions[stock_id] = []
        trade_returns_per_stock[stock_id] = []

    if recommendation == "Comprar":
        # Open (or add to) position
        open_positions[stock_id].append(cotation)

    elif recommendation == "Vender":
        # Close all open positions at current price
        if len(open_positions[stock_id]) > 0:
            mean_buy_price = np.mean(open_positions[stock_id])
            trade_return = (cotation - mean_buy_price) / mean_buy_price
            trade_returns_per_stock[stock_id].append(trade_return)
            open_positions[stock_id] = []

# Close any remaining open positions at last available price
for stock_id in open_positions:
    if len(open_positions[stock_id]) > 0:
        mean_buy_price = np.mean(open_positions[stock_id])
        last_price = last_cotation_per_stock[stock_id]  # assumes last cotation is last observed
        trade_return = (last_price - mean_buy_price) / mean_buy_price
        trade_returns_per_stock[stock_id].append(trade_return)
        open_positions[stock_id] = []

# # --- Compute cumulative return per stock ---
# cumulative_return_per_stock = {}

# for stock_id, returns in trade_returns_per_stock.items():
#     if len(returns) == 0:
#         cumulative_return_per_stock[stock_id] = 0.0
#     else:
#         cumulative_return_per_stock[stock_id] = (
#             np.prod([1 + r for r in returns]) - 1
#         )

# # --- Compute total portfolio cumulative return ---
# total_cumulative_return = (
#     np.prod([1 + r for r in cumulative_return_per_stock.values()]) - 1
# )

# --- Print results ---
total_trade_return = 0
for stock_id, cr in trade_returns_per_stock.items():
    total_trade_return += sum(cr)
    print(stock_id, f"{sum(cr) * 100:.2f}%")

print("TOTAL", f"{total_trade_return * 100:.2f}%")

ALUP11 4.63%
AURE3 18.75%
CPLE3 25.34%
ENEV3 115.18%
EQTL3 20.08%
TOTAL 183.98%


### Buy and Hold

In [79]:
import numpy as np

cotation_per_stock = {}
pregao = {}

for i, result in enumerate(results):
    stock_id = result["ACAO"]
    if stock_id not in cotation_per_stock:
        cotation_per_stock[stock_id] = []
        pregao[stock_id] = []
    cotation = result["PRECO_ULTIMO_NEGOCIO"]
    pregao[stock_id] += [result["DATA_DO_PREGAO"]]

    cotation_per_stock[stock_id] += [cotation]

# Buy-and-hold cumulative return per stock
buy_and_hold_cr = {}
trade_returns_per_stock = {}

for stock_id, prices in cotation_per_stock.items():
    # Safety check
    if len(prices) < 2:
        continue

    first_price = prices[0]
    last_price = prices[-1]

    buy_and_hold_cr[stock_id] = (last_price / first_price) - 1
    trade_returns_per_stock[stock_id] = (last_price - first_price) / first_price


# Equal-weighted portfolio cumulative return
total_buy_and_hold_cr = np.mean(list(buy_and_hold_cr.values()))


# # Printing results
# print("Buy-and-Hold Cumulative Return per Stock:")
# for stock_id, cr in buy_and_hold_cr.items():
#     print(stock_id, f"{cr * 100:.2f}%")


# print("\nBuy-and-Hold Total Cumulative Return (Equal-Weighted Portfolio):")
# print(f"{total_buy_and_hold_cr * 100:.2f}%")

# Printing results
print("Buy-and-Hold Trade Return per Stock:")
total_trade_return = 0
for stock_id, cr in trade_returns_per_stock.items():
    total_trade_return += cr
    print(stock_id, f"{cr * 100:.2f}%")


print("\nBuy-and-Hold Total Trade Return:")
print(f"{total_trade_return * 100:.2f}%")

Buy-and-Hold Trade Return per Stock:
ALUP11 12.25%
AURE3 -8.12%
CPLE3 46.87%
ENEV3 54.63%
EQTL3 12.29%

Buy-and-Hold Total Trade Return:
117.91%


### Price-Only Manager

In [80]:
import json
import numpy as np

FOLDER = "/Users/thiagocastroferreira/Desktop/kubernetes/results/baseline"

with open(f"{FOLDER}/decisions_sample.json") as f:
    decisions = json.load(f)

with open(f"{FOLDER}/results_sample.json") as f:
    results = json.load(f)

# Store open positions and realized trade returns
open_positions = {}
trade_returns_per_stock = {}
last_cotation_per_stock = {}

for i, decision in enumerate(decisions):
    stock_id = decision["stock_id"]
    analysis_date = decision["analysis_date"]
    recommendation = decision["recommendation"]

    cotation = [
        r for r in results if r["ACAO"] == stock_id and r["DATA_DO_PREGAO"] == analysis_date
    ][0]["PRECO_ULTIMO_NEGOCIO"]
    last_cotation_per_stock[stock_id] = cotation

    if stock_id not in open_positions:
        open_positions[stock_id] = []
        trade_returns_per_stock[stock_id] = []

    if recommendation == "Comprar":
        # Open (or add to) position
        open_positions[stock_id].append(cotation)

    elif recommendation == "Vender":
        # Close all open positions at current price
        if len(open_positions[stock_id]) > 0:
            mean_buy_price = np.mean(open_positions[stock_id])
            trade_return = (cotation - mean_buy_price) / mean_buy_price
            trade_returns_per_stock[stock_id].append(trade_return)
            open_positions[stock_id] = []

# Close any remaining open positions at last available price
for stock_id in open_positions:
    if len(open_positions[stock_id]) > 0:
        mean_buy_price = np.mean(open_positions[stock_id])
        last_price = last_cotation_per_stock[stock_id]  # assumes last cotation is last observed
        trade_return = (last_price - mean_buy_price) / mean_buy_price
        trade_returns_per_stock[stock_id].append(trade_return)
        open_positions[stock_id] = []

# # --- Compute cumulative return per stock ---
# cumulative_return_per_stock = {}

# for stock_id, returns in trade_returns_per_stock.items():
#     if len(returns) == 0:
#         cumulative_return_per_stock[stock_id] = 0.0
#     else:
#         cumulative_return_per_stock[stock_id] = (
#             np.prod([1 + r for r in returns]) - 1
#         )

# # --- Compute total portfolio cumulative return ---
# total_cumulative_return = (
#     np.prod([1 + r for r in cumulative_return_per_stock.values()]) - 1
# )

# --- Print results ---
total_trade_return = 0
for stock_id, cr in trade_returns_per_stock.items():
    total_trade_return += sum(cr)
    print(stock_id, f"{sum(cr) * 100:.2f}%")

print("TOTAL", f"{total_trade_return * 100:.2f}%")

ALUP11 17.99%
AURE3 24.34%
CPLE3 0.00%
EQTL3 0.00%
ENEV3 33.38%
TOTAL 75.71%
